# Capstone 1 — Fraud Detection Agent with RAG

**An agentic fraud-investigation system on AWS, driven from Azure Databricks.**

This single notebook delivers the full capstone pipeline:

1. **Read** a batch of card transactions from the Delta table.
2. **Train & deploy** an XGBoost fraud classifier to a **SageMaker real-time endpoint**.
3. **Build a Bedrock Knowledge Base** over the `fraud_rules/` corpus (RAG) and validate retrieval.
4. **Wire an agent loop** (LangGraph + Claude Sonnet 4.5) that exposes four tools and *chooses its own
   sequence*: `invoke_fraud_model`, `retrieve_rules`, `get_customer_profile`, `generate_report`.
5. **Run a batch** end-to-end → one structured investigation report per high-risk transaction, with a
   **CloudWatch audit trail** of every tool call.
6. **(Stretch)** Orchestrate the whole batch as an **MWAA Airflow DAG**.

> **How to run:** set your `STUDENT_NUM` in Section 0, then run top to bottom. Long-running cells
> (training ~4 min, endpoint deploy ~6 min, KB sync) are clearly marked. Each section is idempotent
> where practical, so you can re-run pieces without redoing everything.

---

### Architecture at a glance

```
 Delta table (fad_transactions)
        │  Spark read
        ▼
 [Feature engineering] ──► train XGBoost (SageMaker Training Job)
        │                              │ model.tar.gz in S3
        │                              ▼
        │                     SageMaker real-time endpoint
        ▼                              ▲
 sample batch ──► score all ──► high-risk subset
                                       │
                                       ▼
                      ┌──────────  AGENT LOOP  ──────────┐
                      │  (LangGraph + Claude Sonnet 4.5) │
                      │   chooses tool order each turn   │
                      │                                  │
                      │  invoke_fraud_model ─► endpoint  │
                      │  retrieve_rules ─────► Bedrock KB (RAG over fraud_rules/)
                      │  get_customer_profile ─► customers Delta
                      │  generate_report ────► structured JSON → S3
                      └───────────────┬──────────────────┘
                                      │  every tool call logged
                                      ▼
                          CloudWatch Logs (audit trail)
```


### Step 0 — install agent / RAG libraries first

On Databricks, `%pip install` **restarts the Python interpreter**, so run this cell **before** anything else (the auth cell below recreates all clients afterward). `langchain-aws` / `langgraph` are usually pre-installed on the course cluster — this is a safety net.

In [0]:
# Run this FIRST. %pip restarts the Python kernel on Databricks, so any state
# created before it would be lost. Nothing important runs above this cell.
%pip install -q langchain langchain-aws langgraph 


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
#Additional serverless requirements
#%pip install -q sagemaker

# Section 0 — Environment Setup

The standard auth cell (Databricks → AWS via `boto3`). Replace `STUDENT_NUM` with your 2-digit number.
We extend the guidance cell with the extra clients this capstone needs (`s3`, `sts`, `logs`,
`cloudwatch`, `mwaa`) and a SageMaker SDK session bound to our `boto3` session (so the Python SDK works
from Databricks, not just from inside SageMaker Studio).

In [0]:
# --- Standard auth cell (from Capstone_1_Guidance.md) -----------------------
import boto3, os, json, time

STUDENT_NUM = "50"  # <-- your number
AWS_ACCOUNT_ID   = "962804699607"

scope = f"aws-course-creds-{STUDENT_NUM}"
os.environ["AWS_ACCESS_KEY_ID"]     = dbutils.secrets.get(scope, "aws-access-key-id")
os.environ["AWS_SECRET_ACCESS_KEY"] = dbutils.secrets.get(scope, "aws-secret-access-key")
os.environ["AWS_REGION"] = os.environ["AWS_DEFAULT_REGION"] = "us-west-2"

session           = boto3.Session(region_name="us-west-2")
sagemaker_client  = session.client("sagemaker")
sagemaker_runtime = session.client("sagemaker-runtime")
bedrock_runtime   = session.client("bedrock-runtime")
bedrock_agent     = session.client("bedrock-agent")
bedrock_agent_rt  = session.client("bedrock-agent-runtime")

# --- Extra clients this capstone needs -------------------------------------
s3         = session.client("s3")
sts        = session.client("sts")
logs       = session.client("logs")
cloudwatch = session.client("cloudwatch")
mwaa       = session.client("mwaa")

CALLER_ARN = sts.get_caller_identity()["Arn"]
print("Caller:", CALLER_ARN)


Caller: arn:aws:iam::962804699607:user/student-50


In [0]:
# --- Project configuration -------------------------------------------------
# Region + models (per the guidance).
AWS_REGION       = "us-west-2"
AGENT_MODEL_ID   = "us.anthropic.claude-sonnet-4-5-20250929-v1:0"   # agent backbone (inference profile)
EMBED_MODEL_ID   = "amazon.titan-embed-text-v2:0"                   # KB embeddings


# Shared course resources (read ids from the shared secret scope where available).
def _shared(key, default=None):
    try:
        return dbutils.secrets.get("aws-course-shared", key)
    except Exception:
        return default

SHARED_BUCKET = _shared("course-s3-bucket", "bread-academy-shared")
S3_PREFIX     = f"student-{STUDENT_NUM}"          # everything we write lands here
KB_ID         = _shared("knowledge-base-id", None) # pre-built fraud-rules KB (Option A)

# SageMaker execution role the Training Job / endpoint assume. The course provisions one;
# pull it from the shared scope, else paste the ARN your instructor gave you.
SAGEMAKER_ROLE_ARN = _shared(
    "sagemaker-execution-role-arn",
    "arn:aws:iam::962804699607:role/bread-academy-sagemaker-execution-role",  # <-- set if not in scope
)

# Unique, per-student names so nothing collides with other students.
ENDPOINT_NAME = f"fraud-clf-student-{STUDENT_NUM}"
LOG_GROUP     = f"/bread-academy/capstone1/student-{STUDENT_NUM}"

# SageMaker Python SDK session bound to OUR boto3 session (Databricks-friendly).
import sagemaker
sm_sdk_session = sagemaker.Session(boto_session=session)

print("Region          :", AWS_REGION)
print("Shared bucket   :", SHARED_BUCKET)
print("S3 work prefix  :", f"s3://{SHARED_BUCKET}/{S3_PREFIX}/")
print("Endpoint name   :", ENDPOINT_NAME)
print("Pre-built KB id :", KB_ID)
print("SageMaker role  :", SAGEMAKER_ROLE_ARN)
print("Agent model     :", AGENT_MODEL_ID)


sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/spark-19db9520-2599-47ae-81d8-19/.config/sagemaker/config.yaml
Region          : us-west-2
Shared bucket   : [REDACTED]
S3 work prefix  : s3://[REDACTED]/student-50/
Endpoint name   : fraud-clf-student-50
Pre-built KB id : [REDACTED]
SageMaker role  : [REDACTED]
Agent model     : us.anthropic.claude-sonnet-4-5-20250929-v1:0


# Section 1 — Load the data from Unity Catalog

Everything lives under `bread_academy.course_data`. We read it with Spark, then pull the transactions
into pandas for XGBoost. We also load `customers` (the agent's `get_customer_profile` tool reads this)
and `ft_fraud_cases` (case-level context for richer reports).

In [0]:
# Transactions: one row per card authorization, ~3% confirmed fraud (label_type_cd = 1).
fad_transactions_spark_df = spark.read.table("bread_academy.course_data.fad_transactions")
print("transactions:", fad_transactions_spark_df.count(), "rows")
fad_transactions_spark_df.groupBy("label_type_cd").count().show()

# Customers: one row per account_num; risk_tier derived from real fraud history.
customers_spark_df = spark.read.table("bread_academy.course_data.customers")
print("customers:", customers_spark_df.count(), "rows")

# Confirmed-fraud case detail (optional, used to enrich reports).
try:
    ft_fraud_cases_spark_df = spark.read.table("bread_academy.course_data.ft_fraud_cases")
    print("fraud cases:", ft_fraud_cases_spark_df.count(), "rows")
except Exception as e:
    ft_fraud_cases_spark_df = None
    print("ft_fraud_cases not available:", e)

# Pull to pandas for training / lookups.
fad_transactions_df = fad_transactions_spark_df.toPandas()
customers_df = customers_spark_df.toPandas()
ft_fraud_cases_df = ft_fraud_cases_spark_df.toPandas() if ft_fraud_cases_spark_df is not None else None

print("\npandas transactions shape:", fad_transactions_df.shape)
fad_transactions_df.head(3)


transactions: 50000 rows
+-------------+-----+
|label_type_cd|count|
+-------------+-----+
|            1| 1500|
|            0|48500|
+-------------+-----+

customers: 5000 rows
fraud cases: 1500 rows

pandas transactions shape: (50000, 34)


,transaction_id,account_num,transaction_ts,tran_amt,tran_cd,merch_cat_code_cd,mrch_nm,merch_city_nm,card_prsn_cd,entry_mode_ind,keyed_swiped_ind,mrch_cntry_cd,merch_zip_cd,card_zip_cd,ecom_in,device_model_cd,ip_address_ipv4_id,old_fraud_score,new_fraud_score,score_type_cd,total_velocity_amt,cash_velocity_amt,hour_24_cnt,cvv2_cvc2_otcm_cd,addr_vrfc_otcm_cd,avail_credit_amt,crdt_line_amt,perc_cred_limt_utlz_pct,nmbr_days_dlnq_cnt,time_on_books_cnt,risk_reason_cd,label_type_cd,label_type_desc,partition_date
0,txn_00049897,acct_0002002,2026-02-23 13:12:07,44.99,25,4900,Southern Company,"Bridgeport, CT",N,ecom,K,US,63390,15585,Y,POS-Ingenico,208.212.237.195,9,64,VAA,69.36,20.53,4,P,N,1066.77,7616.74,0.949,63,133,excessive refunds in short time,0,GENUINE,2026-02-23
1,txn_00049898,acct_0000564,2026-03-30 03:24:00,17.46,27,4900,Reliable Power Co.,"Accra, GH",N,ecom,K,GH,49974,26334,Y,"iPhone15,3",156.208.29.86,476,459,FALCON,17.43,0.59,5,U,A,498.21,23425.39,0.485,83,55,cross-border transaction volume spike,0,GENUINE,2026-03-30
2,txn_00049899,acct_0002725,2026-02-04 14:57:15,12.79,05,5812,The Urban Fork,"Dakar, SN",Y,swipe,S,SN,22742,81144,N,"iPhone15,3",158.207.127.149,255,246,VAA,116.57,8.04,4,U,Z,4678.89,14107.44,0.115,22,218,frequent changes in shipping address,0,GENUINE,2026-02-04


# Section 2 — Feature engineering

XGBoost on SageMaker expects **CSV, target in column 0, numeric only, no header**. We build a single
deterministic `featurize()` function and use it for **both training and inference** so the feature
vector the endpoint sees at scoring time is identical to what it was trained on.

Features we derive from the real Fiserv FAD columns:

| Feature | Source |
|---|---|
| `tran_amt`, `new_fraud_score`, `total_velocity_amt`, `cash_velocity_amt`, `hour_24_cnt` | numeric, as-is |
| `card_not_present` | `card_prsn_cd == 'N'` |
| `foreign_country` | `mrch_cntry_cd != 'US'` |
| `high_risk_mcc` | `merch_cat_code_cd` in a high-risk MCC set (gambling, crypto, …) |
| `cvv_fail`, `avs_fail` | non-match CVV / AVS outcome codes |
| `entry_mode__*` | one-hot of `entry_mode_ind` over a fixed category list |
| `txn_hour_from_ts`, `odd_hour_txn` | parsed from `transaction_ts` (fallback: `hour_24_cnt`) |




In [0]:
import numpy as np
import pandas as pd

# Fixed vocabularies so train- and inference-time encodings line up exactly.
ENTRY_MODES    = ["chip", "swipe", "contactless", "ecom", "manual", "token"]
HIGH_RISK_MCCS = {7995, 6051, 6211, 4829, 7273, 5967, 7801}  # gambling, crypto, wires, etc.
NUMERIC_COLS   = ["tran_amt", "new_fraud_score", "total_velocity_amt",
                  "cash_velocity_amt", "hour_24_cnt"]
TARGET_COL     = "label_type_cd"

def _to_num(s):
    return pd.to_numeric(s, errors="coerce").fillna(0.0)

def featurize(frame: pd.DataFrame) -> pd.DataFrame:
    """Deterministic transaction -> numeric feature matrix. No target column."""
    f = pd.DataFrame(index=frame.index)
    for c in NUMERIC_COLS:
        f[c] = _to_num(frame[c]) if c in frame else 0.0

    cp = frame["card_prsn_cd"].astype(str).str.upper() if "card_prsn_cd" in frame else ""
    f["card_not_present"] = (cp == "N").astype(int)

    cc = frame["mrch_cntry_cd"].astype(str).str.upper() if "mrch_cntry_cd" in frame else "US"
    f["foreign_country"] = (cc != "US").astype(int)

    mcc = _to_num(frame["merch_cat_code_cd"]).astype(int) if "merch_cat_code_cd" in frame else 0
    f["merch_cat_code_cd"] = mcc
    f["high_risk_mcc"] = mcc.isin(HIGH_RISK_MCCS).astype(int)

    # CVV / AVS: treat anything that is not an explicit match ('M') as a fail signal.
    cvv = frame["cvv2_cvc2_otcm_cd"].astype(str).str.upper() if "cvv2_cvc2_otcm_cd" in frame else ""
    avs = frame["addr_vrfc_otcm_cd"].astype(str).str.upper() if "addr_vrfc_otcm_cd" in frame else ""
    f["cvv_fail"] = (~cvv.isin(["M", "MATCH", "Y"])).astype(int)
    f["avs_fail"] = (~avs.isin(["M", "MATCH", "Y"])).astype(int)

    em = frame["entry_mode_ind"].astype(str).str.lower() if "entry_mode_ind" in frame else ""
    for mode in ENTRY_MODES:
        f[f"entry_mode__{mode}"] = (em == mode).astype(int)

    # Parse transaction timestamp into hour and flag odd-hour activity.
    fallback_hour = (_to_num(frame["hour_24_cnt"]) if "hour_24_cnt" in frame
                     else pd.Series(0, index=frame.index))
    fallback_hour = fallback_hour.round().clip(0, 23)

    if "transaction_ts" in frame:
        parsed_ts = pd.to_datetime(frame["transaction_ts"], errors="coerce")
        txn_hour = parsed_ts.dt.hour.fillna(fallback_hour).astype(int)
    else:
        txn_hour = fallback_hour.astype(int)

    f["txn_hour_from_ts"] = txn_hour
    f["odd_hour_txn"] = txn_hour.isin([0, 1, 2, 3, 4, 5]).astype(int)

    return f

# Build the feature frame + the canonical column order used everywhere downstream.
X_all = featurize(fad_transactions_df)
FEATURE_COLUMNS = list(X_all.columns)
y_all = _to_num(fad_transactions_df[TARGET_COL]).astype(int)

print(f"{len(FEATURE_COLUMNS)} features:")
print(FEATURE_COLUMNS)
print("\nfraud rate:", round(float(y_all.mean()), 4))
X_all.head(3)


19 features:
['tran_amt', 'new_fraud_score', 'total_velocity_amt', 'cash_velocity_amt', 'hour_24_cnt', 'card_not_present', 'foreign_country', 'merch_cat_code_cd', 'high_risk_mcc', 'cvv_fail', 'avs_fail', 'entry_mode__chip', 'entry_mode__swipe', 'entry_mode__contactless', 'entry_mode__ecom', 'entry_mode__manual', 'entry_mode__token', 'txn_hour_from_ts', 'odd_hour_txn']

fraud rate: 0.03


,tran_amt,new_fraud_score,total_velocity_amt,cash_velocity_amt,hour_24_cnt,card_not_present,foreign_country,merch_cat_code_cd,high_risk_mcc,cvv_fail,avs_fail,entry_mode__chip,entry_mode__swipe,entry_mode__contactless,entry_mode__ecom,entry_mode__manual,entry_mode__token,txn_hour_from_ts,odd_hour_txn
0,44.99,64,69.36,20.53,4,1,0,4900,0,1,1,0,0,0,1,0,0,13,0
1,17.46,459,17.43,0.59,5,1,1,4900,0,1,1,0,0,0,1,0,0,3,1
2,12.79,246,116.57,8.04,4,0,1,5812,0,1,1,0,1,0,0,0,0,14,0


## Train / Validation / Holdout Split (60 / 20 / 20)

We split the data into three sets so model training and evaluation stay unbiased:

- **Train (60%)**: fit model parameters.
- **Validation (20%)**: tune decisions like early stopping and hyperparameters.
- **Holdout (20%)**: final, untouched evaluation.

### Why This Matters

- Prevents leakage from evaluating on seen data.
- Produces more realistic performance metrics.
- Reduces overfitting to tuning choices by keeping a clean final test set.

In [0]:
from sklearn.model_selection import train_test_split

# Train / validation / holdout (60 / 20 / 20), stratified on the rare label.

X_train, X_tmp, y_train, y_tmp = train_test_split(
    X_all, y_all, test_size=0.40, random_state=42, stratify=y_all)
X_val, X_holdout, y_val, y_holdout = train_test_split(
    X_tmp, y_tmp, test_size=0.50, random_state=42, stratify=y_tmp)

print(f"train={len(X_train)}  val={len(X_val)}  holdout={len(X_holdout)}")
print("fraud per split:",
      round(y_train.mean(),3), round(y_val.mean(),3), round(y_holdout.mean(),3))

# XGBoost CSV format: target FIRST, numeric only, NO header, NO index.
train_csv = pd.concat([y_train.rename("y"), X_train.reset_index(drop=True)
                       .set_index(y_train.index)], axis=1)
val_csv   = pd.concat([y_val.rename("y"),   X_val.reset_index(drop=True)
                       .set_index(y_val.index)],   axis=1)

import os
os.makedirs("/tmp/cap1", exist_ok=True)
train_csv.to_csv("/tmp/cap1/train.csv", header=False, index=False)
val_csv.to_csv("/tmp/cap1/validation.csv", header=False, index=False)

# Class imbalance handle for XGBoost.
neg, pos = int((y_train == 0).sum()), int((y_train == 1).sum())
SCALE_POS_WEIGHT = round(neg / max(pos, 1), 2)
print("scale_pos_weight:", SCALE_POS_WEIGHT)


train=30000  val=10000  holdout=10000
fraud per split: 0.03 0.03 0.03


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ /databricks/python/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3526 in         │
│ run_code                                                                                         │
│                                                                                                  │
│   3523 │   │   │   │   if async_:                                                                │
│   3524 │   │   │   │   │   await eval(code_obj, self.user_global_ns, self.user_ns)               │
│   3525 │   │   │   │   else:                                                                     │
│ ❱ 3526 │   │   │   │   │   exec(code_obj, self.user_global_ns, self.user_ns)                     │
│   3527 │   │   │   finally:                                                                      │
│   3528 │   │   │   │   # Reset our crash handler in place                                        │
│   3529 │   │   │   │   sys.excepthook = old_excepthook                                           │
│                                                                                                  │
│ in <module>:22                                                                                   │
│                                                                                                  │
│   19                                                                                             │
│   20 import os                                                                                   │
│   21 os.makedirs("/tmp/cap1", exist_ok=True)                                                     │
│ ❱ 22 train_csv.to_csv("/tmp/cap1/train.csv", header=False, index=False)                          │
│   23 val_csv.to_csv("/tmp/cap1/validation.csv", header=False, index=False)                       │
│   24                                                                                             │
│   25 # Class imbalance handle for XGBoost.                                                       │
│                                                                                                  │
│ /local_disk0/.ephemeral_nfs/cluster_libraries/python/lib/python3.11/site-packages/pandas/util/_d │
│ ecorators.py:333 in wrapper                                                                      │
│                                                                                                  │
│   330 │   │   │   │   │   FutureWarning,                                                         │
│   331 │   │   │   │   │   stacklevel=find_stack_level(),                                         │
│   332 │   │   │   │   )                                                                          │
│ ❱ 333 │   │   │   return func(*args, **kwargs)                                                   │
│   334 │   │                                                                                      │
│   335 │   │   # error: "Callable[[VarArg(Any), KwArg(Any)], Any]" has no                         │
│   336 │   │   # attribute "__signature__"                                                        │
│                                                                                                  │
│ /local_disk0/.ephemeral_nfs/cluster_libraries/python/lib/python3.11/site-packages/pandas/core/ge │
│ neric.py:3989 in to_csv                                                                          │
│                                                                                                  │
│    3986 │   │   │   decimal=decimal,                                                             │
│    3987 │   │   )                                                                                │
│    3988 │   │                                                                                    │
│ ❱  3989 │   │   return DataFrameRenderer(formatter).to_csv(

In [0]:
# Upload the training channels to our per-student S3 prefix.
train_key = f"{S3_PREFIX}/sagemaker/train/train.csv"
val_key   = f"{S3_PREFIX}/sagemaker/validation/validation.csv"
s3.upload_file("/tmp/cap1/train.csv",      SHARED_BUCKET, train_key)
s3.upload_file("/tmp/cap1/validation.csv", SHARED_BUCKET, val_key)
print("uploaded:")
print(" ", f"s3://{SHARED_BUCKET}/{train_key}")
print(" ", f"s3://{SHARED_BUCKET}/{val_key}")


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ /databricks/python/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3526 in         │
│ run_code                                                                                         │
│                                                                                                  │
│   3523 │   │   │   │   if async_:                                                                │
│   3524 │   │   │   │   │   await eval(code_obj, self.user_global_ns, self.user_ns)               │
│   3525 │   │   │   │   else:                                                                     │
│ ❱ 3526 │   │   │   │   │   exec(code_obj, self.user_global_ns, self.user_ns)                     │
│   3527 │   │   │   finally:                                                                      │
│   3528 │   │   │   │   # Reset our crash handler in place                                        │
│   3529 │   │   │   │   sys.excepthook = old_excepthook                                           │
│                                                                                                  │
│ in <module>:4                                                                                    │
│                                                                                                  │
│   1 # Upload the training channels to our per-student S3 prefix.                                 │
│   2 train_key = f"{S3_PREFIX}/sagemaker/train/train.csv"                                         │
│   3 val_key   = f"{S3_PREFIX}/sagemaker/validation/validation.csv"                               │
│ ❱ 4 s3.upload_file("/tmp/cap1/train.csv",      SHARED_BUCKET, train_key)                         │
│   5 s3.upload_file("/tmp/cap1/validation.csv", SHARED_BUCKET, val_key)                           │
│   6 print("uploaded:")                                                                           │
│   7 print(" ", f"s3://{SHARED_BUCKET}/{train_key}")                                              │
│                                                                                                  │
│ /local_disk0/.ephemeral_nfs/cluster_libraries/python/lib/python3.11/site-packages/botocore/conte │
│ xt.py:123 in wrapper                                                                             │
│                                                                                                  │
│   120 │   │   │   with start_as_current_context():                                               │
│   121 │   │   │   │   if hook:                                                                   │
│   122 │   │   │   │   │   hook()                                                                 │
│ ❱ 123 │   │   │   │   return func(*args, **kwargs)                                               │
│   124 │   │                                                                                      │
│   125 │   │   return wrapper                                                                     │
│   126                                                                                            │
│                                                                                                  │
│ /local_disk0/.ephemeral_nfs/cluster_libraries/python/lib/python3.11/site-packages/boto3/s3/injec │
│ t.py:175 in upload_file                                                                          │
│                                                                                                  │
│   172 │   │   transfer.                                                                          │
│   173 │   """                                                                                    │
│   174 │   with S3Transfer(self, Config) as transfer:                                             │
│ ❱ 175 │   │   return transfer.upload_file(                 

# Section 3 — Train the fraud classifier on SageMaker

We train the managed **XGBoost 1.5-1** container as a SageMaker Training Job (the Week 6 pattern), tuned
for the rare-fraud signal via `scale_pos_weight` and AUC early-stopping. Runs ~3–5 minutes.

> *Optional:* uncomment the `HyperparameterTuner` block at the bottom to run a Bayesian sweep instead of
> a single fit — left off by default to respect the hackathon clock.

In [0]:
'''
from sagemaker.estimator import Estimator
from sagemaker.inputs import TrainingInput
from sagemaker.image_uris import retrieve

xgb_image = retrieve("xgboost", AWS_REGION, version="1.5-1")
print("XGBoost container:", xgb_image)

output_path = f"s3://{SHARED_BUCKET}/{S3_PREFIX}/sagemaker/models"

# Single-fit setup: define one Estimator with fixed infrastructure + algorithm.
# This is NOT a tuning sweep; we run one training job with one parameter set.
xgb = Estimator(
    image_uri=xgb_image,
    role=SAGEMAKER_ROLE_ARN,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    output_path=output_path,
    sagemaker_session=sm_sdk_session,
    base_job_name=f"cap1-fraud-{STUDENT_NUM}",
)

# Single-fit hyperparameters: one explicit configuration used for this run.
# If tuning is enabled (optional cell below), those ranges override this idea.
xgb.set_hyperparameters(
    objective="binary:logistic",
    eval_metric="auc",
    num_round=200,
    max_depth=5,
    eta=0.2,
    gamma=4,
    min_child_weight=6,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=SCALE_POS_WEIGHT,
    early_stopping_rounds=15,
)

train_input = TrainingInput(f"s3://{SHARED_BUCKET}/{train_key}", content_type="text/csv")
val_input   = TrainingInput(f"s3://{SHARED_BUCKET}/{val_key}",   content_type="text/csv")

# One call to fit() launches one SageMaker Training Job using train + validation.
# Validation is used for eval metric tracking and early stopping in that SAME job.
print("Starting training job (3-5 min)...")
xgb.fit({"train": train_input, "validation": val_input}, wait=True, logs="None")

# Save the model artifact URI produced by this single fit run.
MODEL_DATA = xgb.model_data
print("\nModel artifact:", MODEL_DATA)
'''

XGBoost container: 246618743249.dkr.ecr.us-west-2.amazonaws.com/sagemaker-xgboost:1.5-1
Starting training job (3-5 min)...

2026-06-16 19:29:45 Starting - Starting the training job.
2026-06-16 19:29:59 Starting - Preparing the instances for training....
2026-06-16 19:30:23 Downloading - Downloading input data.

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:136)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:133)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:133)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:728)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:446)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:446)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can

In [0]:
# Can be used for fine tuning..
# ---- (OPTIONAL) Bayesian hyperparameter tuning -- uncomment to use --------

# from sagemaker.tuner import HyperparameterTuner, IntegerParameter, ContinuousParameter
# tuner = HyperparameterTuner(
#     estimator=xgb,
#     objective_metric_name="validation:auc",
#     objective_type="Maximize",
#     max_jobs=12, max_parallel_jobs=2, strategy="Bayesian",
#     hyperparameter_ranges={
#         "max_depth": IntegerParameter(3, 10),
#         "eta": ContinuousParameter(0.01, 0.3),
#         "subsample": ContinuousParameter(0.5, 0.9),
#         "colsample_bytree": ContinuousParameter(0.5, 0.9),
#         "min_child_weight": IntegerParameter(1, 10),
#         "gamma": ContinuousParameter(0, 5),
#     },
#     base_tuning_job_name=f"cap1-tune-{STUDENT_NUM}",
# )
# tuner.fit({"train": train_input, "validation": val_input}, wait=True, logs=False)
# MODEL_DATA = tuner.best_estimator().model_data
# print("Best model artifact:", MODEL_DATA)


In [0]:
#copied from Single-fit setup and can be used for hyperparameter tuning as well
from sagemaker.estimator import Estimator
from sagemaker.inputs import TrainingInput
from sagemaker.image_uris import retrieve

xgb_image = retrieve("xgboost", AWS_REGION, version="1.5-1")
print("XGBoost container:", xgb_image)

output_path = f"s3://{SHARED_BUCKET}/{S3_PREFIX}/sagemaker/models"
train_input = TrainingInput(f"s3://{SHARED_BUCKET}/{train_key}", content_type="text/csv")
val_input   = TrainingInput(f"s3://{SHARED_BUCKET}/{val_key}",   content_type="text/csv")

XGBoost container: 246618743249.dkr.ecr.us-west-2.amazonaws.com/sagemaker-xgboost:1.5-1


In [0]:
# =============================================================================
# LAUNCH HYPERPARAMETER TUNING JOB
# =============================================================================
from sagemaker.tuner import HyperparameterTuner, IntegerParameter, ContinuousParameter
print("Configuring hyperparameter tuning job...")
print("=" * 60)

# Create a new estimator for tuning (clone the previous one)
xgb_tuning_estimator = Estimator(
    image_uri=xgb_image,
    role=SAGEMAKER_ROLE_ARN,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    output_path=output_path,
    sagemaker_session=sm_sdk_session,
    base_job_name=f"cap1-fraud-{STUDENT_NUM}"
)

# Set static hyperparameters (ones we won't tune)
xgb_tuning_estimator.set_hyperparameters(
    objective='binary:logistic',
    num_round=100,
    eval_metric='auc',
    early_stopping_rounds=10
)

# Define hyperparameter ranges to explore
hyperparameter_ranges = {
    'max_depth': IntegerParameter(3, 10),           # Tree depth
    'eta': ContinuousParameter(0.01, 0.3),          # Learning rate
    'subsample': ContinuousParameter(0.5, 0.9),     # Sample fraction
    'colsample_bytree': ContinuousParameter(0.5, 0.9),  # Feature fraction
    'min_child_weight': IntegerParameter(1, 10),    # Minimum child weight
    'gamma': ContinuousParameter(0, 5)              # Min loss reduction
}

print("✓ Hyperparameter ranges defined:")
for param, range_obj in hyperparameter_ranges.items():
    print(f"  {param:20s}: {range_obj}")

# Create tuner
tuner = HyperparameterTuner(
    estimator=xgb_tuning_estimator,
    objective_metric_name='validation:auc',
    objective_type='Maximize',
    max_jobs=10,                # Total training jobs to try
    max_parallel_jobs=2,        # Run 2 jobs at a time
    hyperparameter_ranges=hyperparameter_ranges,
    strategy='Bayesian',        # Smart search strategy
    base_tuning_job_name=f"cap1-fraud-{STUDENT_NUM}"
)

print("\n✓ Tuner configured")
print(f"  Strategy:      Bayesian optimization")
print(f"  Max jobs:      20 training jobs")
print(f"  Parallel jobs: 2 simultaneous")
print(f"  Objective:     Maximize validation:auc")

# Launch tuning job (runs in background)
print("\n" + "=" * 60)
print("🚀 Launching hyperparameter tuning job...")
print("=" * 60)

tuner.fit(
    inputs={
        'train': train_input,
        'validation': val_input
    },
    wait=True,     # Don't block - job runs in background
    logs=False      # Don't stream logs (would be overwhelming with 20 jobs)
)

tuning_job_name = tuner.latest_tuning_job.name

MODEL_DATA = tuner.best_estimator().model_data
print("Best model artifact:", MODEL_DATA)

print(f"\n✅ Tuning job launched: {tuning_job_name}")
print(f"\n⏱️ This job will take approximately 30-60 minutes to complete")
print(f"   (20 training jobs × 3-5 min each ÷ 2 parallel = ~30-50 min)")
print(f"\n🔗 View progress in SageMaker Console:")
print(f"   https://console.aws.amazon.com/sagemaker/home?region={session.region_name}#/hyper-tuning-jobs/{tuning_job_name}")
print(f"\n💡 The job runs asynchronously - you can continue with the next section")

Configuring hyperparameter tuning job...
✓ Hyperparameter ranges defined:
  max_depth           : <sagemaker.parameter.IntegerParameter object at 0x7fb6ce759050>
  eta                 : <sagemaker.parameter.ContinuousParameter object at 0x7fb6d1382f90>
  subsample           : <sagemaker.parameter.ContinuousParameter object at 0x7fb6cdaa3810>
  colsample_bytree    : <sagemaker.parameter.ContinuousParameter object at 0x7fb6cda63c90>
  min_child_weight    : <sagemaker.parameter.IntegerParameter object at 0x7fb6cda60a90>
  gamma               : <sagemaker.parameter.ContinuousParameter object at 0x7fb6cda62910>

✓ Tuner configured
  Strategy:      Bayesian optimization
  Max jobs:      20 training jobs
  Parallel jobs: 2 simultaneous
  Objective:     Maximize validation:auc

🚀 Launching hyperparameter tuning job...
.............................................................................!

2026-06-17 17:50:28 Starting - Preparing the instances for training
2026-06-17 17:50:28 Downloadin

In [0]:
# =============================================================================
# MONITOR TUNING JOB STATUS
# =============================================================================

print("Checking tuning job status...")
print("=" * 60)

# Get current status
tuning_job_details = sm_sdk_session.sagemaker_client.describe_hyper_parameter_tuning_job(
    HyperParameterTuningJobName=tuning_job_name
)

status = tuning_job_details['HyperParameterTuningJobStatus']
print(f"Status: {status}")

# Show progress
if 'TrainingJobStatusCounters' in tuning_job_details:
    counters = tuning_job_details['TrainingJobStatusCounters']
    print(f"\nTraining Job Progress:")
    print(f"  Completed:   {counters.get('Completed', 0)}")
    print(f"  InProgress:  {counters.get('InProgress', 0)}")
    print(f"  Stopped:     {counters.get('Stopped', 0)}")
    print(f"  Failed:      {counters.get('RetryableError', 0) + counters.get('NonRetryableError', 0)}")

# Show best result so far (if any jobs completed)
if 'BestTrainingJob' in tuning_job_details:
    best_job = tuning_job_details['BestTrainingJob']
    print(f"\n✅ Best Training Job So Far:")
    print(f"  Job Name: {best_job['TrainingJobName']}")
    print(f"  Objective: {best_job['FinalHyperParameterTuningJobObjectiveMetric']['Value']:.4f}")
    print(f"\n  Best Hyperparameters:")
    for param, value in best_job['TunedHyperParameters'].items():
        print(f"    {param:20s}: {value}")
else:
    print(f"\n⏳ No completed jobs yet - check back in a few minutes")

# Provide code for later checking
print("\n" + "=" * 60)
print("💡 To check status later, run this code:")
print("=" * 60)
print(f"""
import boto3
sm_client = boto3.client('sagemaker', region_name='{session.region_name}')
details = sm_client.describe_hyper_parameter_tuning_job(
    HyperParameterTuningJobName='{tuning_job_name}'
)
print(f"Status: {{details['HyperParameterTuningJobStatus']}}")
if 'BestTrainingJob' in details:
    best = details['BestTrainingJob']
    print(f"Best AUC: {{best['FinalHyperParameterTuningJobObjectiveMetric']['Value']:.4f}}")
""")

print("\n" + "=" * 60)
print("🎯 What's Happening:")
print("=" * 60)
print("1. Bayesian optimizer selects hyperparameter combinations to try")
print("2. Each combination trains an XGBoost model (3-5 minutes)")
print("3. Two models train in parallel to save time")
print("4. After each job, the optimizer learns and picks better combinations")
print("5. After 20 jobs, the best hyperparameters are identified")
print("\n✓ The optimized model will outperform our manually tuned one!")

Checking tuning job status...
Status: Completed

Training Job Progress:
  Completed:   10
  InProgress:  0
  Stopped:     0
  Failed:      0

✅ Best Training Job So Far:
  Job Name: cap1-fraud-50-260617-1747-001-4cebe97e
  Objective: 0.9887

  Best Hyperparameters:
    colsample_bytree    : 0.5006071393249497
    eta                 : 0.10344366772621955
    gamma               : 1.9669553631720742
    max_depth           : 5
    min_child_weight    : 7
    subsample           : 0.7728892401099245

💡 To check status later, run this code:

import boto3
sm_client = boto3.client('sagemaker', region_name='us-west-2')
details = sm_client.describe_hyper_parameter_tuning_job(
    HyperParameterTuningJobName='cap1-fraud-50-260617-1747'
)
print(f"Status: {details['HyperParameterTuningJobStatus']}")
if 'BestTrainingJob' in details:
    best = details['BestTrainingJob']
    print(f"Best AUC: {best['FinalHyperParameterTuningJobObjectiveMetric']['Value']:.4f}")


🎯 What's Happening:
1. Bayesian opt

In [0]:
import boto3
sm_client = boto3.client('sagemaker', region_name='us-west-2')
details = sm_client.describe_hyper_parameter_tuning_job(
    HyperParameterTuningJobName=tuning_job_name
)
print(f"Status: {details['HyperParameterTuningJobStatus']}")
if 'BestTrainingJob' in details:
    best = details['BestTrainingJob']
    print(f"Best AUC: {best['FinalHyperParameterTuningJobObjectiveMetric']['Value']:.4f}")

Status: Completed
Best AUC: 0.9887


In [0]:
details

{'HyperParameterTuningJobName': 'cap1-fraud-50-260617-1747',
 'HyperParameterTuningJobArn': 'arn:aws:sagemaker:us-west-2:962804699607:hyper-parameter-tuning-job/cap1-fraud-50-260617-1747',
 'HyperParameterTuningJobConfig': {'Strategy': 'Bayesian',
  'HyperParameterTuningJobObjective': {'Type': 'Maximize',
   'MetricName': 'validation:auc'},
  'ResourceLimits': {'MaxNumberOfTrainingJobs': 10,
   'MaxParallelTrainingJobs': 2},
  'ParameterRanges': {'IntegerParameterRanges': [{'Name': 'max_depth',
     'MinValue': '3',
     'MaxValue': '10',
     'ScalingType': 'Auto'},
    {'Name': 'min_child_weight',
     'MinValue': '1',
     'MaxValue': '10',
     'ScalingType': 'Auto'}],
   'ContinuousParameterRanges': [{'Name': 'eta',
     'MinValue': '0.01',
     'MaxValue': '0.3',
     'ScalingType': 'Auto'},
    {'Name': 'subsample',
     'MinValue': '0.5',
     'MaxValue': '0.9',
     'ScalingType': 'Auto'},
    {'Name': 'colsample_bytree',
     'MinValue': '0.5',
     'MaxValue': '0.9',
     'S

# Section 4 — Deploy to a real-time endpoint & evaluate on holdout

We wrap the trained artifact in a `Model` and deploy it to a real-time endpoint (Week 7 pattern). Deploy
takes ~5–8 minutes. Then we score the **holdout** through the live endpoint and report AUC, precision,
recall, and the confusion matrix — proving the endpoint works *and* that the model generalizes.

In [0]:
from sagemaker.model import Model

# If the endpoint already exists from a previous run, reuse it; else deploy.
def endpoint_exists(name):
    try:
        sagemaker_client.describe_endpoint(EndpointName=name)
        return True
    except sagemaker_client.exceptions.ClientError:
        return False

if endpoint_exists(ENDPOINT_NAME):
    print(f"Endpoint {ENDPOINT_NAME} already exists — reusing.")
else:
    fraud_model = Model(
        image_uri=xgb_image,
        model_data=MODEL_DATA,
        role=SAGEMAKER_ROLE_ARN,
        sagemaker_session=sm_sdk_session,
        name=f"cap1-fraud-model-{STUDENT_NUM}-{int(time.time())}",
    )
    print(f"Deploying {ENDPOINT_NAME} (5-8 min)...")
    fraud_model.deploy(
        initial_instance_count=1,
        instance_type="ml.m5.large",
        endpoint_name=ENDPOINT_NAME,
    )
    print("Deployed.")

print("Status:", sagemaker_client.describe_endpoint(EndpointName=ENDPOINT_NAME)["EndpointStatus"])


Deploying fraud-clf-student-50 (5-8 min)...
------!Deployed.
Status: InService


### Endpoint Scoring Helpers

This section defines two reusable methods for sending feature vectors to the deployed SageMaker endpoint.

- `score_features(feat_row)`: scores a **single featurized transaction row** and returns one fraud probability.
  - Used later by the agent tool (`invoke_fraud_model`) when investigating one transaction at a time.
- `score_frame(feat, batch=500)`: scores a **full feature dataframe in mini-batches** and returns probabilities for all rows.
  - Used for holdout/batch evaluation where we need efficient multi-row inference.

Both methods use the same `FEATURE_COLUMNS` order to guarantee inference-time input matches the model's training-time feature layout.

In [0]:
# Low-level invoke helper (CSV in -> probability out). Used by eval AND the agent tool.
def score_features(feat_row: pd.Series) -> float:
    """Score a single featurized row (ordered by FEATURE_COLUMNS) -> fraud probability."""
    csv_line = ",".join(str(float(feat_row[c])) for c in FEATURE_COLUMNS)
    resp = sagemaker_runtime.invoke_endpoint(
        EndpointName=ENDPOINT_NAME, ContentType="text/csv", Body=csv_line)
    out = resp["Body"].read().decode().strip()
    # XGBoost text container returns a bare probability (or CSV of them).
    return float(out.split(",")[0])

# Score the holdout through the live endpoint in mini-batches (CSV multi-row).
def score_frame(feat: pd.DataFrame, batch=500):
    probs = []
    rows = feat[FEATURE_COLUMNS].astype(float).values
    for i in range(0, len(rows), batch):
        chunk = rows[i:i+batch]
        body = "\n".join(",".join(str(v) for v in r) for r in chunk)
        resp = sagemaker_runtime.invoke_endpoint(
            EndpointName=ENDPOINT_NAME, ContentType="text/csv", Body=body)
        out = resp["Body"].read().decode().strip().splitlines()
        probs.extend(float(x.split(",")[0]) for x in out)
    return np.array(probs)

holdout_probs = score_frame(X_holdout)
print("scored holdout:", len(holdout_probs))


scored holdout: 10000


### Holdout Metrics Evaluation

Here we evaluate model performance on the **holdout set** (data never used for fitting or tuning) to estimate real-world generalization.

- **ROC-AUC**: ranking quality across thresholds; higher is better.
- **Precision**: among predicted fraud transactions, how many are truly fraud.
- **Recall**: among true fraud transactions, how many we successfully catch.
- **F1**: harmonic balance of precision and recall.
- **Confusion Matrix**: counts of TN, FP, FN, TP for the selected threshold.

Together, these metrics help us assess the tradeoff between catching fraud and limiting false alarms before moving to production usage.

In [0]:
from sklearn.metrics import roc_auc_score, precision_score, recall_score, confusion_matrix, f1_score

THRESHOLD = 0.5
y_true = y_holdout.values
y_pred = (holdout_probs >= THRESHOLD).astype(int)

print("Holdout evaluation")
print("=" * 40)
print(f"ROC-AUC   : {roc_auc_score(y_true, holdout_probs):.4f}")
print(f"Precision : {precision_score(y_true, y_pred, zero_division=0):.4f}")
print(f"Recall    : {recall_score(y_true, y_pred, zero_division=0):.4f}")
print(f"F1        : {f1_score(y_true, y_pred, zero_division=0):.4f}")
print("\nConfusion matrix [[TN FP][FN TP]]:")
print(confusion_matrix(y_true, y_pred))


Holdout evaluation
ROC-AUC   : 0.9908
Precision : 0.8319
Recall    : 0.6433
F1        : 0.7256

Confusion matrix [[TN FP][FN TP]]:
[[9661   39]
 [ 107  193]]


# Section 5 — Bedrock Knowledge Base (the RAG part)

The agent retrieves matching fraud rules from a **Bedrock Knowledge Base** built over the `fraud_rules/`
corpus (~32 markdown docs, each with YAML frontmatter and worked examples). Two paths:

- **Option A — reuse the pre-built KB** (`bread-academy-fraud-kb`): fastest, `KB_ID` came from the shared
  scope in Section 0. Run the validation cell and move on.
- **Option B — build your own** (more of the learning): upload `fraud_rules/*.md` to your S3 prefix, then
  create a KB in the Bedrock console (Titan v2 embeddings, OpenSearch Serverless quick-create, fixed-size
  ~300-token chunking), **sync**, and set `KB_ID` to your new id.

Either way, our `retrieve_rules` tool is a thin wrapper over `bedrock_agent_rt.retrieve`.

In [0]:


# --- Option B (optional): upload the corpus so you can build your own KB ----
# Skip if you're using the pre-built KB (Option A). Run, then create the KB in the
# Bedrock console pointed at the prefix printed below, sync, and paste its id into KB_ID.

import glob
RULES_GLOB = "fraud_rules/*.md"          # the corpus sits next to the guidance file
files = glob.glob(RULES_GLOB)
print(f"found {len(files)} rule docs")
for f in files:
    key = f"{S3_PREFIX}/fraud_rules/{os.path.basename(f)}"
    s3.upload_file(f, SHARED_BUCKET, key)
if files:
    print(f"uploaded to s3://{SHARED_BUCKET}/{S3_PREFIX}/fraud_rules/")
    print("Now create a KB in the Bedrock console over that prefix, sync, and set KB_ID.")
else:
    print("No local fraud_rules/ found — use Option A (pre-built KB).")


found 32 rule docs
uploaded to s3://[REDACTED]/student-50/fraud_rules/
Now create a KB in the Bedrock console over that prefix, sync, and set KB_ID.


In [0]:
# --- Option B (programmatic): create a Bedrock KB + data source + sync --------
# Adapted from exercises/week_17_rag_fundamentals/bootstrap_and_setup.py
import uuid
import time
from botocore.exceptions import ClientError

acct = sts.get_caller_identity()["Account"]
region = AWS_REGION

# Source docs uploaded in the previous cell.
DOC_PREFIX = f"{S3_PREFIX}/fraud_rules/"
DOC_BUCKET_ARN = f"arn:aws:s3:::{SHARED_BUCKET}"

# S3 Vectors store (same pattern used in bootstrap_and_setup.py)
VECTOR_BUCKET = f"bread-academy-fraud-kb-vectors-{STUDENT_NUM}"
VECTOR_INDEX = "fraud-index"
VECTOR_BUCKET_ARN = f"arn:aws:s3vectors:{region}:{acct}:bucket/{VECTOR_BUCKET}"
VECTOR_INDEX_ARN = f"{VECTOR_BUCKET_ARN}/index/{VECTOR_INDEX}"

# Role used by Bedrock KB. Prefer a dedicated role from shared secrets if present;
# fallback to the SageMaker role configured earlier.
try:
    KB_ROLE_ARN = dbutils.secrets.get("aws-course-shared", "bedrock-kb-role-arn")    
except Exception:
    KB_ROLE_ARN = SAGEMAKER_ROLE_ARN

KB_ROLE_ARN = "arn:aws:iam::962804699607:role/BreadAcademyKBRole"

print("Using KB role:", KB_ROLE_ARN)
print("Docs source   :", f"s3://{SHARED_BUCKET}/{DOC_PREFIX}")

# Create/reuse the S3 Vectors bucket + index.
s3vec = session.client("s3vectors", region_name=region)
try:
    s3vec.create_vector_bucket(vectorBucketName=VECTOR_BUCKET)
    print("Created vector bucket:", VECTOR_BUCKET)
except ClientError as e:
    if e.response["Error"]["Code"] == "ConflictException":
        print("Vector bucket already exists:", VECTOR_BUCKET)
    else:
        raise

try:
    s3vec.create_index(
        vectorBucketName=VECTOR_BUCKET,
        indexName=VECTOR_INDEX,
        dataType="float32",
        dimension=1024,
        distanceMetric="cosine",
    )
    print("Created vector index:", VECTOR_INDEX)
except ClientError as e:
    if e.response["Error"]["Code"] == "ConflictException":
        print("Vector index already exists:", VECTOR_INDEX)
    else:
        raise

# Create KB.
kb_name = f"fraud-rules-kb-student-{STUDENT_NUM}-{uuid.uuid4().hex[:8]}"
embed_arn = f"arn:aws:bedrock:{region}::foundation-model/{EMBED_MODEL_ID}"

kb_resp = bedrock_agent.create_knowledge_base(
    name=kb_name,
    roleArn=KB_ROLE_ARN,
    knowledgeBaseConfiguration={
        "type": "VECTOR",
        "vectorKnowledgeBaseConfiguration": {
            "embeddingModelArn": embed_arn,
            "embeddingModelConfiguration": {
                "bedrockEmbeddingModelConfiguration": {
                    "dimensions": 1024,
                    "embeddingDataType": "FLOAT32",
                }
            },
        },
    },
    storageConfiguration={
        "type": "S3_VECTORS",
        "s3VectorsConfiguration": {
            "vectorBucketArn": VECTOR_BUCKET_ARN,
            "indexArn": VECTOR_INDEX_ARN,
        },
    },
)
KB_ID = kb_resp["knowledgeBase"]["knowledgeBaseId"]
print("Created KB:", KB_ID)

# Wait for KB to become ACTIVE.
while True:
    kb_desc = bedrock_agent.get_knowledge_base(knowledgeBaseId=KB_ID)["knowledgeBase"]
    kb_status = kb_desc["status"]
    print("KB status:", kb_status)
    if kb_status == "ACTIVE":
        break
    if kb_status in ("FAILED", "DELETE_UNSUCCESSFUL"):
        raise RuntimeError(f"KB creation failed with status={kb_status}")
    time.sleep(10)

# Create data source on your uploaded S3 fraud_rules docs.
ds_resp = bedrock_agent.create_data_source(
    knowledgeBaseId=KB_ID,
    name=f"{kb_name}-s3",
    dataSourceConfiguration={
        "type": "S3",
        "s3Configuration": {
            "bucketArn": DOC_BUCKET_ARN,
            "inclusionPrefixes": [DOC_PREFIX],
        },
    },
    vectorIngestionConfiguration={
        "chunkingConfiguration": {
            "chunkingStrategy": "FIXED_SIZE",
            "fixedSizeChunkingConfiguration": {
                "maxTokens": 300,
                "overlapPercentage": 20,
            },
        }
    },
)
DS_ID = ds_resp["dataSource"]["dataSourceId"]
print("Created data source:", DS_ID)

# Sync (ingest) documents.
ing = bedrock_agent.start_ingestion_job(knowledgeBaseId=KB_ID, dataSourceId=DS_ID)
ing_id = ing["ingestionJob"]["ingestionJobId"]
print("Started ingestion job:", ing_id)

while True:
    j = bedrock_agent.get_ingestion_job(
        knowledgeBaseId=KB_ID,
        dataSourceId=DS_ID,
        ingestionJobId=ing_id,
    )["ingestionJob"]
    status = j["status"]
    print("Ingestion status:", status, j.get("statistics", {}))
    if status == "COMPLETE":
        break
    if status in ("FAILED", "STOPPED"):
        raise RuntimeError(f"Ingestion ended with status={status}")
    time.sleep(15)

print("\nKB ready:", KB_ID)
print("You can now run the validation cell below (bedrock_agent_rt.retrieve).")

Using KB role: arn:aws:iam::962804699607:role/BreadAcademyKBRole
Docs source   : s3://[REDACTED]/student-50/fraud_rules/
Vector bucket already exists: bread-academy-fraud-kb-vectors-50
Vector index already exists: fraud-index
Created KB: HVZ7N2IZ5O
KB status: CREATING
KB status: ACTIVE
Created data source: QMMEZZJPFJ
Started ingestion job: RNWUXXRDQZ
Ingestion status: STARTING {'numberOfDocumentsScanned': 0, 'numberOfMetadataDocumentsScanned': 0, 'numberOfNewDocumentsIndexed': 0, 'numberOfModifiedDocumentsIndexed': 0, 'numberOfMetadataDocumentsModified': 0, 'numberOfDocumentsDeleted': 0, 'numberOfDocumentsFailed': 0}
Ingestion status: COMPLETE {'numberOfDocumentsScanned': 32, 'numberOfMetadataDocumentsScanned': 0, 'numberOfNewDocumentsIndexed': 15, 'numberOfModifiedDocumentsIndexed': 0, 'numberOfMetadataDocumentsModified': 0, 'numberOfDocumentsDeleted': 0, 'numberOfDocumentsFailed': 17}

KB ready: HVZ7N2IZ5O
You can now run the validation cell below (bedrock_agent_rt.retrieve).


In [0]:
#KB_ID = "COROQTD4NC"

In [0]:
KB_ID = "HVZ7N2IZ5O"

In [0]:
'''
"""
Option A - reuse the pre-built KB (fastest).** A fraud-rules KB already exists in the
account. Get its id from the shared scope and start querying immediately:
"""
KB_ID = dbutils.secrets.get("aws-course-shared", "knowledge-base-id")  # bread-academy-fraud-kb
#(LXMHVMVY1L)
resp = bedrock_agent_rt.retrieve(
    knowledgeBaseId=KB_ID,
    retrievalQuery={"text": "high velocity card-not-present from a high-risk country"},
)
for r in resp["retrievalResults"]:
    print(r["content"]["text"][:200])

'''

--- rule_id: BF-GEO-020 category: geographic risk severity: medium source: Bread Financial internal --- # Cross-border Card-not-Present from a High-Risk Country Code ## Summary This rule identifies tr
and it falls under a lower-risk MCC and is domestic. ## Severity and Recommended Action - **Severity**: High - **Recommended Action**: Transactions that trigger this rule should be reviewed for potent
--- rule_id: BF-GEO-020 category: geographic risk severity: medium source: Bread Financial internal --- # Cross-border Card-not-Present from a High-Risk Country Code ## Summary This rule identifies tr
A threshold of $100 is used; transactions over this amount may be flagged for further review. - **Merchant Country Code (mrch_cntry_cd)**: - Transactions should primarily originate from the US (domest
**Transaction That Does Not Trigger the Rule**: - **tran_amt**: $120 - **merch_cat_code_cd**: 5541 - **card_prsn_cd**: N - **entry_mode_ind**: ecom - **mrch_cntry_cd**: NG - **new_fraud_score**: 4

In [0]:
# --- Validate retrieval before wiring the KB to the agent -------------------
assert KB_ID, "KB_ID is not set. Use the pre-built KB (Option A) or build your own (Option B)."

def kb_retrieve(query: str, k: int = 4):
    """Raw retrieval against the Bedrock KB -> list of {text, source, score}."""
    resp = bedrock_agent_rt.retrieve(
        knowledgeBaseId=KB_ID,
        retrievalQuery={"text": query},
        retrievalConfiguration={"vectorSearchConfiguration": {"numberOfResults": k}},
    )
    hits = []
    for r in resp.get("retrievalResults", []):
        hits.append({
            "text":   r["content"]["text"],
            "source": r.get("location", {}).get("s3Location", {}).get("uri", "unknown"),
            "score":  round(r.get("score", 0.0), 4),
        })
    return hits

# Sanity-check with a query whose thresholds are wired to the real data.
for h in kb_retrieve("high velocity card-not-present from a high-risk country", k=3):
    print(f"[{h['score']}] {h['source'].split('/')[-1]}")
    print("   ", h["text"][:160].replace("\n", " "), "...\n")


[0.7193] 007_velocity-transaction-count-burst.md
    and it falls under a lower-risk MCC and is domestic. ## Severity and Recommended Action - **Severity**: High - **Recommended Action**: Transactions that trigger ...

[0.713] 016_mcc-baseline-low-risk.md
    A threshold of $100 is used; transactions over this amount may be flagged for further review. - **Merchant Country Code (mrch_cntry_cd)**: - Transactions should ...

[0.6999] 016_mcc-baseline-low-risk.md
    **Transaction That Does Not Trigger the Rule**: - **tran_amt**: $120 - **merch_cat_code_cd**: 5541 - **card_prsn_cd**: N - **entry_mode_ind**: ecom - **mrch_cnt ...



# Section 6 — The four agent tools (with CloudWatch audit trail)

The agent is given four tools and **chooses its own sequence**. Every tool call emits a structured
log line to a per-student CloudWatch log group, so the full reasoning trace is auditable back to the
rules and customer data the agent used.

| Tool | Does |
|---|---|
| `invoke_fraud_model` | look up the transaction, featurize it, call the SageMaker endpoint → risk score |
| `retrieve_rules` | `bedrock_agent_rt.retrieve` against the KB → matching fraud rules |
| `get_customer_profile` | read one row from `customers` |
| `generate_report` | assemble a structured investigation report (JSON) and persist to S3 |


In [0]:
# --- CloudWatch audit-trail plumbing ---------------------------------------
def _ensure_log_group():
    try:
        logs.create_log_group(logGroupName=LOG_GROUP)
    except logs.exceptions.ResourceAlreadyExistsException:
        pass

# We need to UNCOMMENT THIS TO CREATE THE LOG GROUPS
_ensure_log_group()

# Each agent run gets its own log stream (run_id). audit() appends one event per tool call.
_audit_seq = {"token": None}
def start_audit_stream(run_id: str):
    try:
        logs.create_log_stream(logGroupName=LOG_GROUP, logStreamName=run_id)
    except logs.exceptions.ResourceAlreadyExistsException:
        pass
    _audit_seq["token"] = None
    _audit_seq["run_id"] = run_id

def audit(step: str, txn: str, args: dict, result_summary):
    """Emit one structured audit line to CloudWatch for the current run."""
    run_id = _audit_seq.get("run_id", "adhoc")
    event = {
        "timestamp": int(time.time() * 1000),
        "message": json.dumps({
            "step": step, "txn": txn, "args": args,
            "result_summary": result_summary, "ts": time.time(),
        }),
    }
    kw = {"logGroupName": LOG_GROUP, "logStreamName": run_id, "logEvents": [event]}
    if _audit_seq["token"]:
        kw["sequenceToken"] = _audit_seq["token"]
    try:
        resp = logs.put_log_events(**kw)
        _audit_seq["token"] = resp.get("nextSequenceToken")
    except logs.exceptions.InvalidSequenceTokenException as e:
        # Recover the expected token and retry once.
        import re
        m = re.search(r"sequenceToken is: (\S+)", str(e))
        if m:
            kw["sequenceToken"] = m.group(1)
            resp = logs.put_log_events(**kw)
            _audit_seq["token"] = resp.get("nextSequenceToken")

print("CloudWatch audit trail ready ->", LOG_GROUP)


CloudWatch audit trail ready -> /bread-academy/capstone1/student-50


In [0]:
# --- Fast in-memory indexes the tools read from ----------------------------
# Transaction lookup by id (so a tool can featurize a single row on demand).
fad_transactions_df_indexed = fad_transactions_df.set_index("transaction_id", drop=False)
customers_df_indexed = customers_df.set_index("customer_id", drop=False)
ft_fraud_cases_df_indexed = (ft_fraud_cases_df.set_index("transaction_id", drop=False)
                             if ft_fraud_cases_df is not None else None)
print("indexed:", len(fad_transactions_df_indexed), "transactions,", len(customers_df_indexed), "customers")


indexed: 50000 transactions, 5000 customers


In [0]:
from langchain_core.tools import tool

REPORTS = {}   # transaction_id -> report dict, collected this session

@tool
def invoke_fraud_model(transaction_id: str) -> str:
    """Score a transaction for fraud risk using the deployed SageMaker model.

    Returns the model's fraud probability (0-1) plus the key risk signals
    (amount, card-present flag, country, velocity) for the given transaction_id
    (format txn_00000001). Call this FIRST to decide whether deeper investigation
    is warranted."""
    if transaction_id not in fad_transactions_df_indexed.index:
        return json.dumps({"error": f"transaction {transaction_id} not found"})
    row = fad_transactions_df_indexed.loc[[transaction_id]]
    score = float(score_features(featurize(row).iloc[0]))
    r = row.iloc[0]
    out = {
        "transaction_id": transaction_id,
        "risk_score": round(score, 4),
        "tran_amt": float(_to_num(pd.Series([r.get("tran_amt", 0)]))[0]),
        "card_present": str(r.get("card_prsn_cd", "")),
        "country": str(r.get("mrch_cntry_cd", "")),
        "mcc": str(r.get("merch_cat_code_cd", "")),
        "merchant": str(r.get("mrch_nm", "")),
        "issuer_fraud_score": str(r.get("new_fraud_score", "")),
    }
    audit("invoke_fraud_model", transaction_id, {"transaction_id": transaction_id},
          {"risk_score": out["risk_score"]})
    return json.dumps(out)

@tool
def retrieve_rules(query: str) -> str:
    """Retrieve the fraud-policy rules that match a transaction pattern (RAG).

    Pass a natural-language description of the suspicious pattern (e.g.
    'high velocity card-not-present purchase from a high-risk country at a
    crypto merchant'). Returns the most relevant fraud rules from the knowledge
    base, with their source documents. Use this to ground your decision in
    written policy rather than guessing."""
    hits = kb_retrieve(query, k=4)
    audit("retrieve_rules", _audit_seq.get("run_id", ""), {"query": query},
          {"n_hits": len(hits), "top_source": hits[0]["source"].split("/")[-1] if hits else None})
    return json.dumps([{"source": h["source"].split("/")[-1], "score": h["score"],
                        "rule": h["text"][:700]} for h in hits])

@tool
def get_customer_profile(account_num: str) -> str:
    """Look up the customer profile for an account (context for the investigation).

    Pass the account_num (format acct_0000001). Returns tenure, average monthly
    spend, credit band, risk tier, delinquency flag, occupation, segment, and a
    human-readable profile summary. Use this to judge whether a transaction is
    anomalous for THIS customer."""
    if account_num not in customers_df_indexed.index:
        audit("get_customer_profile", account_num, {"account_num": account_num}, {"found": False})
        return json.dumps({"error": f"account {account_num} not found"})
    c = customers_df_indexed.loc[account_num].to_dict()
    keep = ["customer_id", "account_tenure_months", "avg_monthly_spend", "home_zip",
            "credit_score_band", "risk_tier", "delinquency_flag", "occupation",
            "segment", "profile_summary"]
    prof = {k: (str(c.get(k)) if c.get(k) is not None else None) for k in keep}
    audit("get_customer_profile", account_num, {"account_num": account_num},
          {"risk_tier": prof.get("risk_tier")})
    return json.dumps(prof)

@tool
def generate_report(transaction_id: str, account_num: str, risk_score: float,
                    decision: str, matched_rules: str, customer_context: str,
                    rationale: str, recommended_action: str) -> str:
    """Write the final structured investigation report for a transaction.

    Call this LAST, once you have the risk score, the matching rules, and the
    customer context. decision must be one of ESCALATE / MONITOR / CLEAR.
    matched_rules and customer_context are short summaries of what you found.
    The report is persisted to S3 and returned as JSON."""
    report = {
        "transaction_id": transaction_id,
        "account_num": account_num,
        "risk_score": round(float(risk_score), 4),
        "decision": decision.upper().strip(),
        "matched_rules": matched_rules,
        "customer_context": customer_context,
        "rationale": rationale,
        "recommended_action": recommended_action,
        "generated_ts": pd.Timestamp.utcnow().isoformat(),
        "run_id": _audit_seq.get("run_id", ""),
    }
    key = f"{S3_PREFIX}/reports/{transaction_id}.json"
    s3.put_object(Bucket=SHARED_BUCKET, Key=key,
                  Body=json.dumps(report, indent=2).encode())
    REPORTS[transaction_id] = report
    audit("generate_report", transaction_id,
          {"decision": report["decision"]},
          {"s3": f"s3://{SHARED_BUCKET}/{key}"})
    return json.dumps(report)

AGENT_TOOLS = [invoke_fraud_model, retrieve_rules, get_customer_profile, generate_report]
print("tools ready:", [t.name for t in AGENT_TOOLS])


tools ready: ['invoke_fraud_model', 'retrieve_rules', 'get_customer_profile', 'generate_report']


# Section 7 — The agent loop (LangGraph + Claude Sonnet 4.5)

Reason, Act, and Iterate - ReAct Agent

A single ReAct agent loop. We hand the four tools to `create_react_agent` and let the model decide the
order — look up the risk score, retrieve the rules that match what it sees, pull the customer profile if
context is needed, and finally write the report. **A fixed script is not an agent; the model picks the
sequence.**

In [0]:
from langchain_aws import ChatBedrockConverse
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.prebuilt import create_react_agent

llm = ChatBedrockConverse(
    model=AGENT_MODEL_ID,
    region_name=AWS_REGION,
    temperature=0.0,
    max_tokens=2048,
)

AGENT_SYSTEM_PROMPT = (
    "You are a senior fraud investigator at a card issuer. For each transaction you receive:\n"
    "1. Call invoke_fraud_model to get the model risk score and the transaction's key signals.\n"
    "2. Based on those signals, call retrieve_rules with a focused description of the suspicious\n"
    "   pattern to find the fraud policies that apply.\n"
    "3. Call get_customer_profile (using the account_num) to judge whether this is anomalous for\n"
    "   this customer.\n"
    "4. Finally call generate_report exactly once with: a decision of ESCALATE, MONITOR, or CLEAR;\n"
    "   short summaries of the matched rules and customer context; a rationale that cites the\n"
    "   specific rules and signals; and a concrete recommended_action.\n"
    "Decide ESCALATE for clear policy violations or high model risk, MONITOR for ambiguous cases,\n"
    "CLEAR when the activity is consistent with the customer's profile. Be concise and evidence-driven."
)

fraud_agent = create_react_agent(model=llm, tools=AGENT_TOOLS, prompt=AGENT_SYSTEM_PROMPT)
print("agent ready with tools:", [t.name for t in AGENT_TOOLS])


/home/spark-19db9520-2599-47ae-81d8-19/.ipykernel/54994/command-7823490448588764-144825956:26: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  fraud_agent = create_react_agent(model=llm, tools=AGENT_TOOLS, prompt=AGENT_SYSTEM_PROMPT)


agent ready with tools: ['invoke_fraud_model', 'retrieve_rules', 'get_customer_profile', 'generate_report']


In [0]:
def investigate(transaction_id: str, verbose: bool = True) -> dict:
    """Run one full agent investigation for a transaction. Returns the report dict."""
    account_num = str(fad_transactions_df_indexed.loc[transaction_id, "account_num"])
    run_id = f"student-{STUDENT_NUM}-{transaction_id}-{int(time.time())}"
    start_audit_stream(run_id)

    task = (f"Investigate transaction {transaction_id} (account {account_num}). "
            f"Decide whether it is fraudulent and write the investigation report.")
    result = fraud_agent.invoke({"messages": [HumanMessage(content=task)]})

    if verbose:
        used = [m.tool_calls[0]["name"] for m in result["messages"]
                if getattr(m, "tool_calls", None)]
        print(f"{transaction_id}: tool sequence -> {used}")
    return REPORTS.get(transaction_id, {"transaction_id": transaction_id,
                                         "decision": "UNKNOWN",
                                         "note": result["messages"][-1].content[:300]})

# Smoke-test the agent on one known-fraud transaction.
_sample_fraud = fad_transactions_df_indexed[fad_transactions_df_indexed["label_type_cd"] == 1]["transaction_id"].iloc[0]
demo_report = investigate(_sample_fraud)
print("\nDEMO REPORT")
print(json.dumps(demo_report, indent=2)[:1200])


2026/06/17 19:50:28 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: 3 validation errors for ChatMessage
content.str
  Input should be a valid string [type=string_type, input_value=[{'type': 'text', 'text':...SRlB7EiwE3aEYJZbMlVpt'}], input_type=list]
    For further information visit https://errors.pydantic.dev/2.13/v/string_type
content.list[tagged-union[TextContentPart,ImageContentPart,AudioContentPart]].1
  Input tag 'tool_use' found using 'type' does not match any of the expected tags: 'text', 'image_url', 'input_audio' [type=union_tag_invalid, input_value={'type': 'tool_use', 'nam...PCfQpFFhv98nwsYJsfj8MQ'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/union_tag_invalid
content.list[tagged-union[TextContentPart,ImageContentPart,AudioContentPart]].2
  Input tag 'tool_use' found using 'type' does not match any of the expected tags: 'text', 'image_url', 'input_audio' [type=union_tag_invalid, input_value

txn_00049905: tool sequence -> ['invoke_fraud_model', 'retrieve_rules', 'generate_report']

DEMO REPORT
{
  "transaction_id": "txn_00049905",
  "account_num": "acct_0003146",
  "risk_score": 0.5836,
  "decision": "ESCALATE",
  "matched_rules": "Transaction matches high-risk country rule (RS - Serbia listed in BF-GEO-018) for card-not-present transactions. MCC 4814 (prepaid/telecom services) from Serbia triggers geographic mismatch policy. Multiple rules flag RS as high-risk jurisdiction requiring enhanced scrutiny.",
  "customer_context": "High-risk subprime customer (227 months tenure) with poor credit, delinquency flag active, avg monthly spend $191. Warehouse worker with financial stress indicators. Transaction amount ($18.84) is within normal spend range but geographic pattern (Serbia CNP) is highly anomalous for US-based customer (ZIP 59694).",
  "rationale": "Model risk score of 0.5836 (moderate-high) combined with multiple policy violations warrants escalation. Transaction origi

Trace(request_id=tr-b5e20887f5054b29b6c2550787a63d2c)

# Section 8 — Run the batch end-to-end

The production shape: **score the whole batch → filter the high-risk subset → investigate each with the
agent → collect one report per flagged transaction.** We score a sample batch through the endpoint, take
everything at or above `HIGH_RISK_THRESHOLD`, and let the agent investigate each one.

In [0]:
HIGH_RISK_THRESHOLD = 0.7
BATCH_SIZE          = 300     # sample-batch size to score
N_CASES_PER_GROUP   = 10      # number of transactions per option
RANDOM_SEED         = 42

# Take a sample batch (mix in known fraud so the batch has high-risk cases to find).
fraud_ids = fad_transactions_df_indexed[fad_transactions_df_indexed["label_type_cd"] == 1]["transaction_id"]
rng = np.random.RandomState(7)
batch_ids = list(pd.Index(rng.choice(fad_transactions_df_indexed["transaction_id"].values,
                                     size=BATCH_SIZE, replace=False))
                 .union(pd.Index(fraud_ids.sample(min(20, len(fraud_ids)),
                                                   random_state=7).values)))
batch = fad_transactions_df_indexed.loc[batch_ids]

# Score the whole batch through the endpoint in one pass.
batch_scores = score_frame(featurize(batch))
batch = batch.assign(risk_score=batch_scores)

# Build the three investigation cohorts you asked for.
top_risky_df = batch.sort_values("risk_score", ascending=False).head(N_CASES_PER_GROUP)
least_risky_df = batch.sort_values("risk_score", ascending=True).head(N_CASES_PER_GROUP)
random_df = batch.sample(n=min(N_CASES_PER_GROUP, len(batch)), random_state=RANDOM_SEED)

# New option: 50/50 blend (half high-risk + half low-risk).
half_n = N_CASES_PER_GROUP // 2
blend_high = batch.sort_values("risk_score", ascending=False).head(half_n)
blend_low = batch.sort_values("risk_score", ascending=True).head(half_n)
blend_50_50_df = pd.concat([blend_high, blend_low], axis=0)
blend_50_50_df = blend_50_50_df.drop_duplicates(subset=["transaction_id"]).head(N_CASES_PER_GROUP)

# If overlap reduced rows, top up from remaining random rows.
if len(blend_50_50_df) < N_CASES_PER_GROUP:
    needed = N_CASES_PER_GROUP - len(blend_50_50_df)
    remaining = batch[~batch["transaction_id"].isin(blend_50_50_df["transaction_id"])]
    if len(remaining) > 0:
        top_up = remaining.sample(n=min(needed, len(remaining)), random_state=RANDOM_SEED)
        blend_50_50_df = pd.concat([blend_50_50_df, top_up], axis=0)

# Keep previous high-risk summary for compatibility with downstream cells.
high_risk = batch[batch["risk_score"] >= HIGH_RISK_THRESHOLD].sort_values("risk_score", ascending=False)
print(f"batch scored: {len(batch)}  |  high-risk (>= {HIGH_RISK_THRESHOLD}): {len(high_risk)}")

print("\nTop 10 most risky:")
print(top_risky_df[["transaction_id", "account_num", "risk_score", "label_type_cd"]].to_string(index=False))

print("\nTop 10 least risky:")
print(least_risky_df[["transaction_id", "account_num", "risk_score", "label_type_cd"]].to_string(index=False))

print("\n10 random transactions:")
print(random_df[["transaction_id", "account_num", "risk_score", "label_type_cd"]].to_string(index=False))

print("\n50/50 blend (high + low):")
print(blend_50_50_df[["transaction_id", "account_num", "risk_score", "label_type_cd"]].to_string(index=False))

INVESTIGATION_POOLS = {
    "top_risky": top_risky_df,
    "least_risky": least_risky_df,
    "random": random_df,
    "blend_50_50": blend_50_50_df,
}


batch scored: 320  |  high-risk (>= 0.7): 20

Top 10 most risky:
transaction_id  account_num  risk_score  label_type_cd
  txn_00031344 acct_0003758    0.992525              1
  txn_00025390 acct_0002966    0.986863              1
  txn_00008497 acct_0003860    0.967438              1
  txn_00035788 acct_0003113    0.963027              1
  txn_00034012 acct_0002627    0.961382              1
  txn_00021982 acct_0000941    0.955547              1
  txn_00007940 acct_0004332    0.939490              1
  txn_00038456 acct_0004773    0.936625              0
  txn_00047784 acct_0004216    0.919238              1
  txn_00006200 acct_0002375    0.917351              1

Top 10 least risky:
transaction_id  account_num  risk_score  label_type_cd
  txn_00009602 acct_0002282    0.001219              0
  txn_00045613 acct_0004115    0.001245              0
  txn_00024240 acct_0000721    0.001263              0
  txn_00008320 acct_0002122    0.001263              0
  txn_00024815 acct_0001535    0.0

In [0]:
# Investigate transactions by selectable cohort.
# Options: "top_risky", "least_risky", "random", "blend_50_50", or "all"
RUN_GROUP = "blend_50_50"

if RUN_GROUP == "all":
    selected_groups = ["top_risky", "least_risky", "random", "blend_50_50"]
else:
    selected_groups = [RUN_GROUP]

batch_reports = []

for group_name in selected_groups:
    group_df = INVESTIGATION_POOLS[group_name]
    txn_ids = group_df["transaction_id"].tolist()
    print(f"\nInvestigating group '{group_name}' ({len(txn_ids)} transactions)...")

    for txn in txn_ids:
        rep = investigate(txn, verbose=True)
        rep["group"] = group_name
        batch_reports.append(rep)

print("\nINVESTIGATION SUMMARY")
print("=" * 90)
for r in batch_reports:
    print(f"[{r.get('group','?')}] {r['transaction_id']}  {str(r.get('decision','?')):8s}  "
          f"risk={r.get('risk_score','?')}  -> {str(r.get('recommended_action',''))[:60]}")



Investigating group 'blend_50_50' (10 transactions)...


2026/06/17 19:50:58 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: 3 validation errors for ChatMessage
content.str
  Input should be a valid string [type=string_type, input_value=[{'type': 'text', 'text':...9fy4bVn0CZLDR2mifF85n'}], input_type=list]
    For further information visit https://errors.pydantic.dev/2.13/v/string_type
content.list[tagged-union[TextContentPart,ImageContentPart,AudioContentPart]].1
  Input tag 'tool_use' found using 'type' does not match any of the expected tags: 'text', 'image_url', 'input_audio' [type=union_tag_invalid, input_value={'type': 'tool_use', 'nam...nK4w1VUUR1D6HwdkfhliPY'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/union_tag_invalid
content.list[tagged-union[TextContentPart,ImageContentPart,AudioContentPart]].2
  Input tag 'tool_use' found using 'type' does not match any of the expected tags: 'text', 'image_url', 'input_audio' [type=union_tag_invalid, input_value

txn_00031344: tool sequence -> ['invoke_fraud_model', 'retrieve_rules', 'generate_report']


2026/06/17 19:51:29 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: 3 validation errors for ChatMessage
content.str
  Input should be a valid string [type=string_type, input_value=[{'type': 'text', 'text':...r8o43LOmr6VHbrYi2ZVe6'}], input_type=list]
    For further information visit https://errors.pydantic.dev/2.13/v/string_type
content.list[tagged-union[TextContentPart,ImageContentPart,AudioContentPart]].1
  Input tag 'tool_use' found using 'type' does not match any of the expected tags: 'text', 'image_url', 'input_audio' [type=union_tag_invalid, input_value={'type': 'tool_use', 'nam...caHPPDnfQ8NABIHpsDD1j7'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/union_tag_invalid
content.list[tagged-union[TextContentPart,ImageContentPart,AudioContentPart]].2
  Input tag 'tool_use' found using 'type' does not match any of the expected tags: 'text', 'image_url', 'input_audio' [type=union_tag_invalid, input_value

txn_00025390: tool sequence -> ['invoke_fraud_model', 'retrieve_rules', 'generate_report']


...

2026/06/17 19:54:36 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: 3 validation errors for ChatMessage
content.str
  Input should be a valid string [type=string_type, input_value=[{'type': 'text', 'text':...6zInyEI4d2P8P9sk7d3sw'}], input_type=list]
    For further information visit https://errors.pydantic.dev/2.13/v/string_type
content.list[tagged-union[TextContentPart,ImageContentPart,AudioContentPart]].1
  Input tag 'tool_use' found using 'type' does not match any of the expected tags: 'text', 'image_url', 'input_audio' [type=union_tag_invalid, input_value={'type': 'tool_use', 'nam...kZ4uzDBOkkloXklWJ00KEN'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/union_tag_invalid
content.list[tagged-union[TextContentPart,ImageContentPart,AudioContentPart]].2
  Input tag 'tool_use' found using 'type' does not match any of the expected tags: 'text', 'image_url', 'input_audio' [type=union_tag_invalid, input_value

txn_00008320: tool sequence -> ['invoke_fraud_model', 'retrieve_rules', 'generate_report']


2026/06/17 19:55:00 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: 3 validation errors for ChatMessage
content.str
  Input should be a valid string [type=string_type, input_value=[{'type': 'text', 'text':...rx2Zi6QR44ycEkz1XLuxA'}], input_type=list]
    For further information visit https://errors.pydantic.dev/2.13/v/string_type
content.list[tagged-union[TextContentPart,ImageContentPart,AudioContentPart]].1
  Input tag 'tool_use' found using 'type' does not match any of the expected tags: 'text', 'image_url', 'input_audio' [type=union_tag_invalid, input_value={'type': 'tool_use', 'nam...6Aw1YH0VPGsMMzz5kIk4Zp'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/union_tag_invalid
content.list[tagged-union[TextContentPart,ImageContentPart,AudioContentPart]].2
  Input tag 'tool_use' found using 'type' does not match any of the expected tags: 'text', 'image_url', 'input_audio' [type=union_tag_invalid, input_value

txn_00024815: tool sequence -> ['invoke_fraud_model', 'retrieve_rules', 'generate_report']

INVESTIGATION SUMMARY
[blend_50_50] txn_00031344  ESCALATE  risk=0.9925  -> Immediately decline transaction and contact cardholder via v
[blend_50_50] txn_00025390  ESCALATE  risk=0.9869  -> Decline transaction immediately and contact cardholder via v
[blend_50_50] txn_00008497  ESCALATE  risk=0.9674  -> ESCALATE to fraud operations immediately. Block card and con
[blend_50_50] txn_00035788  ESCALATE  risk=0.963  -> Immediately block card and contact cardholder at verified ph
[blend_50_50] txn_00034012  ESCALATE  risk=0.9614  -> ESCALATE to fraud operations immediately. Block card and con
[blend_50_50] txn_00009602  CLEAR     risk=0.0012  -> CLEAR transaction for processing. No further investigation r
[blend_50_50] txn_00045613  CLEAR     risk=0.0012  -> Clear transaction for processing. No further investigation r
[blend_50_50] txn_00024240  CLEAR     risk=0.0013  -> Clear transaction for proces

[Trace(request_id=tr-7d58008c35c44019a66972f89c5a6c31), Trace(request_id=tr-84094068ae1a429895c47a6dd5c845ae), Trace(request_id=tr-868ae46cfe4b4827878e711965130a8b), Trace(request_id=tr-118abb911f3d4ae09cf4aaf9a7af0f00), Trace(request_id=tr-9c7a99b28a62442e85ab30e95a885691), Trace(request_id=tr-5f788e78ed4c445aadc63acc7c643174), Trace(request_id=tr-f71b7de76f89457caa553f2060920370), Trace(request_id=tr-9bbd0a911f15403eb37e4b3e851b155b), Trace(request_id=tr-aa9314e90e1f48ecb84be6859d551474), Trace(request_id=tr-146a4837e7234dcf8faeb2ace672164b)]

In [0]:
# Persist the batch summary alongside the per-transaction reports already in S3.
run_ts = pd.Timestamp.utcnow()
run_ts_key = run_ts.strftime("%Y%m%dT%H%M%SZ")

summary = {
    "student": STUDENT_NUM,
    "batch_size": int(len(batch)),
    "high_risk_count": int(len(high_risk)),
    "investigated": len(batch_reports),
    "decisions": pd.Series([r.get("decision") for r in batch_reports]).value_counts().to_dict(),
    "reports": batch_reports,
    "generated_ts": run_ts.isoformat(),
}
summary_key = f"{S3_PREFIX}/reports/_batch_summary_{run_ts_key}.json"
s3.put_object(Bucket=SHARED_BUCKET, Key=summary_key,
              Body=json.dumps(summary, indent=2).encode())
print("batch summary ->", f"s3://{SHARED_BUCKET}/{summary_key}")
print("decisions:", summary["decisions"])


batch summary -> s3://[REDACTED]/student-50/reports/_batch_summary_20260617T195601Z.json
decisions: {'ESCALATE': 5, 'CLEAR': 5}


In [0]:
# Read back the CloudWatch audit trail for the most recent investigation (proof of the trace).
last_run = _audit_seq.get("run_id")
print("Audit trail for run:", last_run, "\n")
ev = logs.get_log_events(logGroupName=LOG_GROUP, logStreamName=last_run,
                         startFromHead=True)["events"]
for e in ev:
    m = json.loads(e["message"])
    print(f"  {m['step']:22s} txn={m.get('txn','')}  {json.dumps(m['result_summary'])}")


Audit trail for run: student-50-txn_00024815-1781726096 

  get_customer_profile   txn=acct_0001535  {"risk_tier": "high"}
  invoke_fraud_model     txn=txn_00024815  {"risk_score": 0.0013}
  retrieve_rules         txn=student-50-txn_00024815-1781726096  {"n_hits": 4, "top_source": "016_mcc-baseline-low-risk.md"}
  generate_report        txn=txn_00024815  {"s3": "s3://[REDACTED]/student-50/reports/txn_00024815.json"}


# Deliverables checklist & wrap-up

What this notebook produced:

- ✅ **Trained fraud model deployed to a SageMaker endpoint**, evaluated on a stratified holdout
  (AUC / precision / recall / confusion matrix) — Sections 3–4.
- ✅ **Bedrock Knowledge Base over `fraud_rules/`**, retrieval validated on sample queries — Section 5.
- ✅ **Agent loop wiring all four tools** (`invoke_fraud_model`, `retrieve_rules`,
  `get_customer_profile`, `generate_report`); the model chooses its own sequence — Sections 6–7.
- ✅ **One structured investigation report per high-risk transaction** in a sample batch, persisted to
  `s3://{bucket}/student-NN/reports/` — Section 8.
- ✅ **CloudWatch audit trail** of every tool call, per run — Sections 6 & 8.

All artifacts live under your `student-{NN}/` prefix in the course S3 bucket (`reports/`, `airflow/`,
`sagemaker/`) and in the CloudWatch log group `/bread-academy/capstone1/student-{NN}`.

In [0]:
# Quick inventory of everything written to S3 under your prefix.
print(f"Artifacts under s3://{SHARED_BUCKET}/{S3_PREFIX}/\n")
token = None
while True:
    kw = {"Bucket": SHARED_BUCKET, "Prefix": f"{S3_PREFIX}/"}
    if token: kw["ContinuationToken"] = token
    resp = s3.list_objects_v2(**kw)
    for o in resp.get("Contents", []):
        print(f"  {o['Size']:>9d}  {o['Key']}")
    if not resp.get("IsTruncated"): break
    token = resp["NextContinuationToken"]


Artifacts under s3://[REDACTED]/student-50/

       2980  student-50/fraud_rules/001_aml-ctr-10k-aggregation.md
       3270  student-50/fraud_rules/002_aml-structuring-detection.md
       3091  student-50/fraud_rules/003_aml-sar-suspicious-patterns.md
       3689  student-50/fraud_rules/004_aml-funnel-account-behavior.md
       2657  student-50/fraud_rules/005_aml-high-risk-jurisdiction-monitoring.md
       2760  student-50/fraud_rules/006_velocity-24h-amount-spike.md
       3075  student-50/fraud_rules/007_velocity-transaction-count-burst.md
       2857  student-50/fraud_rules/008_velocity-cash-advance-rate.md
       3431  student-50/fraud_rules/009_velocity-cross-channel.md
       3112  student-50/fraud_rules/010_velocity-new-account-ramp.md
       3710  student-50/fraud_rules/011_mcc-gambling-7995.md
       3192  student-50/fraud_rules/012_mcc-quasi-cash-crypto-6051.md
       3449  student-50/fraud_rules/013_mcc-money-transfer-4829.md
       2846  student-50/fraud_rules/014_mcc-jewe

## Cleanup (cost control)

The real-time endpoint bills per hour while it exists. **Delete it when you're done** (keep it if you
still want to demo the agent). The KB, S3 artifacts, and CloudWatch logs are cheap to leave in place;
remove them too if your instructor asks.

In [0]:
# --- Delete the SageMaker endpoint (uncomment to run) ----------------------
# sagemaker_client.delete_endpoint(EndpointName=ENDPOINT_NAME)
# try:
#     cfg = sagemaker_client.describe_endpoint_config  # endpoint config + model linger; remove if desired
# except Exception:
#     pass
# print(f"Deleted endpoint {ENDPOINT_NAME}")

# --- Pause your Airflow DAG so it does not get retriggered (optional) -------
# _mwaa_rest("PATCH", f"/dags/{DAG_ID}", {"is_paused": True})
# print(f"Paused DAG {DAG_ID}")

print("Cleanup cell ready — uncomment the lines above to tear down billable resources.")
